# Vie-GameEmo — Stage 0: Chuẩn bị Dataset

**Notebook này thực hiện:**
1. Tải video từ YouTube (hoặc dùng video có sẵn)
2. Tiền xử lý: tách audio, trích xuất frames, phát hiện webcam
3. Gán nhãn đa tác tử: Whisper ASR → OpenFace AUs → Qwen-VL → Qwen-Audio → Consolidator
4. Xuất annotations JSON ra Kaggle output

**Yêu cầu Kaggle:**
- Accelerator: **GPU T4 x1** (hoặc P100)
- Internet: **BẬT** (Settings → Internet)
- Runtime: ~4-6 giờ cho 50 clips

---
⚠️ **Stage 0 chạy độc lập.** Sau khi xong, download `data/annotations/` và dùng cho notebook Training.

In [ ]:
# ============================================================
# CELL 1 — Kiểm tra môi trường
# ============================================================
import os, sys, subprocess

IS_KAGGLE = os.path.exists('/kaggle')
WORKING = '/kaggle/working' if IS_KAGGLE else '/tmp/vie-gameemo'
os.makedirs(WORKING, exist_ok=True)
print(f'Platform : {"Kaggle" if IS_KAGGLE else "Local"}')
print(f'Working  : {WORKING}')

# GPU
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f'GPU      : {gpu.name} | {gpu.total_memory / 1e9:.1f} GB VRAM')
else:
    print('⚠️  Không có GPU — hãy bật Accelerator trong Settings')

In [ ]:
# ============================================================
# CELL 2 — CẤU HÌNH (chỉnh tại đây)
# ============================================================

# --- Nguồn dữ liệu ---
# 'youtube'  : tải từ danh sách URL bên dưới
# 'existing' : dùng video có sẵn trong /kaggle/input/<dataset_name>/
DATA_SOURCE = 'youtube'   # <-- ĐỔI TẠI ĐÂY

# Nếu DATA_SOURCE = 'existing', tên Kaggle dataset chứa video
EXISTING_DATASET_PATH = '/kaggle/input/vie-gameemo-videos'  # mount từ Kaggle Dataset

# --- Danh sách URL YouTube (dùng khi DATA_SOURCE = 'youtube') ---
YOUTUBE_URLS = [
    # Thêm URL YouTube livestream/clip của streamer vào đây
    # 'https://www.youtube.com/watch?v=XXXXXXXXX',
    # 'https://www.youtube.com/watch?v=YYYYYYYYY',
]

# --- Model annotation ---
# Chọn model cho Consolidator (annotation reasoning)
# 'Qwen/Qwen2.5-7B-Instruct'   → T4 OK, chất lượng tốt (khuyến nghị)
# 'Qwen/Qwen2.5-1.5B-Instruct' → T4 thoải mái, nhanh hơn ~3x, chất lượng thấp hơn
# 'Qwen/Qwen2.5-32B-Instruct'  → cần A100, chất lượng cao nhất
# 'Qwen/Qwen3-8B'              → Qwen3 mới nhất, hỗ trợ thinking mode, T4 OK (4bit)
ANNOTATION_MODEL = 'Qwen/Qwen2.5-7B-Instruct'   # <-- ĐỔI TẠI ĐÂY

# Model Qwen-VL cho visual descriptions
QWEN_VL_MODEL = 'Qwen/Qwen2.5-VL-7B-Instruct'   # hoặc 'Qwen/Qwen2.5-VL-3B-Instruct'

# Model Qwen-Audio cho audio descriptions  
QWEN_AUDIO_MODEL = 'Qwen/Qwen2-Audio-7B-Instruct'

# Quantization (4bit tiết kiệm VRAM nhất)
QUANTIZATION = '4bit'   # '4bit' | '8bit' | 'none'

# --- Gán nhãn cảm xúc thủ công ---
# Điền nhãn cho từng clip (clip_id: tên file không có extension)
# Nhãn hợp lệ: hype | tilted | focused | disappointed | shocked | amused | neutral
CLIP_LABELS = {
    # 'clip_001': 'hype',
    # 'clip_002': 'tilted',
    # Nếu để trống, notebook tạo placeholder labels để chạy thử
}

# --- Giới hạn số clip (để test nhanh) ---
MAX_CLIPS = 10    # None = không giới hạn

# --- Đường dẫn ---
DATA_DIR     = os.path.join(WORKING, 'data')
RAW_DIR      = os.path.join(DATA_DIR, 'raw_videos')
PROC_DIR     = os.path.join(DATA_DIR, 'processed')
ANNOT_DIR    = os.path.join(DATA_DIR, 'annotations')
PROJECT_DIR  = os.path.join(WORKING, 'vie-gameemo-skeleton')

for d in [DATA_DIR, RAW_DIR, PROC_DIR, ANNOT_DIR]:
    os.makedirs(d, exist_ok=True)

print('Config OK')
print(f'  DATA_SOURCE      : {DATA_SOURCE}')
print(f'  ANNOTATION_MODEL : {ANNOTATION_MODEL}')
print(f'  MAX_CLIPS        : {MAX_CLIPS}')

In [ ]:
# ============================================================
# CELL 3 — Cài thư viện
# ============================================================
# Kaggle có sẵn: torch, numpy, opencv, PIL, ffmpeg
# Cần cài thêm:
!pip install -q \
    transformers>=4.45.0 \
    accelerate>=0.34.0 \
    peft>=0.13.0 \
    bitsandbytes>=0.43.0 \
    faster-whisper>=1.0.3 \
    librosa>=0.10.1 \
    soundfile \
    mediapipe>=0.10.14 \
    yt-dlp>=2024.10.0 \
    pydantic>=2.8.0 \
    pyyaml>=6.0.2 \
    scikit-learn>=1.5.0 \
    qwen-vl-utils

# Thêm Qwen3 support nếu cần
if 'Qwen3' in ANNOTATION_MODEL:
    !pip install -q transformers --upgrade

print('Cài đặt hoàn tất')

In [ ]:
# ============================================================
# CELL 4 — Setup project
# ============================================================
import subprocess, shutil

# Option A: project có sẵn trong Kaggle input dataset (thêm 'vie-gameemo-code' vào notebook inputs)
CODE_INPUT = '/kaggle/input/vie-gameemo-code'

if os.path.exists(CODE_INPUT):
    # Copy sang working (cần write permission)
    if not os.path.exists(PROJECT_DIR):
        shutil.copytree(CODE_INPUT, PROJECT_DIR)
    print(f'Project loaded from Kaggle input: {CODE_INPUT}')

elif os.path.exists('/kaggle/working/vie-gameemo-skeleton'):
    PROJECT_DIR = '/kaggle/working/vie-gameemo-skeleton'
    print(f'Project đã có tại: {PROJECT_DIR}')

else:
    # Option B: clone từ GitHub (cần internet)
    # Thay GITHUB_URL bằng repo thực tế của bạn
    GITHUB_URL = 'https://github.com/YOUR_USERNAME/vie-gameemo-skeleton.git'
    result = subprocess.run(
        ['git', 'clone', '--depth=1', GITHUB_URL, PROJECT_DIR],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print('⚠️  Git clone thất bại. Hãy thêm project code vào Kaggle input dataset.')
        print(result.stderr)
    else:
        print(f'Cloned project → {PROJECT_DIR}')

# Thêm src vào Python path
SRC_DIR = os.path.join(PROJECT_DIR, 'src')
if os.path.exists(SRC_DIR) and SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
    print(f'Added to sys.path: {SRC_DIR}')

# Tạo config.yaml tối giản cho Stage 0
CONFIG_PATH = os.path.join(WORKING, 'config_stage0.yaml')
CONFIG_CONTENT = f"""
seed: 42
logging:
  level: INFO
  file: {WORKING}/logs/stage0.log
  console: true
paths:
  data_root: {DATA_DIR}
  raw_videos: {RAW_DIR}
  processed: {PROC_DIR}
  audios: {PROC_DIR}/audios
  frames: {PROC_DIR}/frames
  faces: {PROC_DIR}/faces
  contexts: {PROC_DIR}/contexts
  annotations: {ANNOT_DIR}
  features: {DATA_DIR}/features
  checkpoints: {WORKING}/checkpoints
  results: {WORKING}/results
annotation:
  openface:
    binary_path: {WORKING}/OpenFace/build/bin/FeatureExtraction
  whisper_asr:
    model_name: openai/whisper-large-v3
    backend: faster-whisper
    compute_type: float16
    language: vi
    vad_filter: true
  qwen_vl:
    model_name: {QWEN_VL_MODEL}
    quantization: {QUANTIZATION}
    max_new_tokens: 200
  qwen_audio:
    model_name: {QWEN_AUDIO_MODEL}
    quantization: {QUANTIZATION}
    max_new_tokens: 200
  consolidator:
    model_name: {ANNOTATION_MODEL}
    quantization: {QUANTIZATION}
    max_new_tokens: 400
  batch_size: 8
preprocess:
  audio:
    sample_rate: 16000
  frames:
    fps: 2
    format: jpg
compute:
  profile: colab
  annotation_vram_gb: 15
"""

os.makedirs(os.path.join(WORKING, 'logs'), exist_ok=True)
with open(CONFIG_PATH, 'w') as f:
    f.write(CONFIG_CONTENT)
print(f'Config → {CONFIG_PATH}')

In [ ]:
# ============================================================
# CELL 5 — Build OpenFace (cần ~15 phút, chạy một lần)
# Bỏ qua nếu đã build hoặc dùng fallback MediaPipe
# ============================================================
OPENFACE_BINARY = os.path.join(WORKING, 'OpenFace/build/bin/FeatureExtraction')
USE_OPENFACE = False  # Sẽ set True nếu build thành công

if os.path.exists(OPENFACE_BINARY):
    USE_OPENFACE = True
    print(f'OpenFace đã có: {OPENFACE_BINARY}')
else:
    print('Building OpenFace từ source (~15 phút)...')
    BUILD_CMDS = [
        'sudo apt-get install -y -q libopencv-dev cmake libboost-all-dev libdlib-dev',
        f'git clone --depth=1 https://github.com/TadasBaltrusaitis/OpenFace.git {WORKING}/OpenFace',
        f'bash {WORKING}/OpenFace/download_models.sh',
        f'mkdir -p {WORKING}/OpenFace/build',
        f'cmake -B {WORKING}/OpenFace/build -S {WORKING}/OpenFace -DCMAKE_BUILD_TYPE=Release -Wno-dev',
        f'cmake --build {WORKING}/OpenFace/build --parallel $(nproc)',
    ]
    success = True
    for cmd in BUILD_CMDS:
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        if r.returncode != 0:
            print(f'⚠️  Lệnh thất bại: {cmd[:60]}...')
            print(r.stderr[-500:] if r.stderr else '')
            success = False
            break

    if success and os.path.exists(OPENFACE_BINARY):
        USE_OPENFACE = True
        print('✅ OpenFace build thành công')
    else:
        print('⚠️  OpenFace build thất bại → dùng MediaPipe làm fallback cho AU extraction')

print(f'USE_OPENFACE = {USE_OPENFACE}')

## Bước 1 — Tải video

In [ ]:
# ============================================================
# CELL 6 — Option A: Tải từ YouTube
# ============================================================
video_paths = []  # list[str] các video đã sẵn sàng

if DATA_SOURCE == 'youtube':
    if not YOUTUBE_URLS:
        print('⚠️  YOUTUBE_URLS rỗng — thêm URL vào CELL 2 và chạy lại')
    else:
        urls = YOUTUBE_URLS[:MAX_CLIPS] if MAX_CLIPS else YOUTUBE_URLS
        for i, url in enumerate(urls):
            clip_id = f'clip_{i+1:03d}'
            out_path = os.path.join(RAW_DIR, f'{clip_id}.mp4')
            if os.path.exists(out_path):
                print(f'  Skip (đã có): {out_path}')
                video_paths.append(out_path)
                continue
            print(f'  Downloading {clip_id}: {url[:60]}...')
            r = subprocess.run([
                'yt-dlp', url,
                '-f', 'best[height<=720]',
                '--merge-output-format', 'mp4',
                '-o', out_path,
                '--no-playlist',
                '--quiet', '--progress',
            ], capture_output=False)
            if r.returncode == 0 and os.path.exists(out_path):
                video_paths.append(out_path)
                print(f'  ✅ {out_path}')
            else:
                print(f'  ❌ Tải thất bại: {url}')

elif DATA_SOURCE == 'existing':
    # Tìm tất cả .mp4 trong dataset input
    if not os.path.exists(EXISTING_DATASET_PATH):
        print(f'⚠️  Không tìm thấy: {EXISTING_DATASET_PATH}')
        print('Hãy thêm dataset video vào notebook inputs (Add data → Your datasets)')
    else:
        import glob
        found = sorted(glob.glob(os.path.join(EXISTING_DATASET_PATH, '**', '*.mp4'), recursive=True))
        if MAX_CLIPS:
            found = found[:MAX_CLIPS]
        # Copy sang working để có thể ghi
        for src in found:
            clip_id = os.path.splitext(os.path.basename(src))[0]
            dst = os.path.join(RAW_DIR, os.path.basename(src))
            if not os.path.exists(dst):
                shutil.copy2(src, dst)
            video_paths.append(dst)
        print(f'Loaded {len(video_paths)} video(s) từ {EXISTING_DATASET_PATH}')

print(f'\nTổng video: {len(video_paths)}')
for p in video_paths[:5]:
    print(f'  {p}')

In [ ]:
# ============================================================
# CELL 7 — Tạo labels CSV
# Điền nhãn thủ công vào CLIP_LABELS ở CELL 2, hoặc để placeholder
# ============================================================
import csv

VALID_LABELS = ['hype', 'tilted', 'focused', 'disappointed', 'shocked', 'amused', 'neutral']
LABELS_CSV = os.path.join(ANNOT_DIR, 'labels.csv')

rows = []
for vp in video_paths:
    clip_id = os.path.splitext(os.path.basename(vp))[0]
    label = CLIP_LABELS.get(clip_id, 'neutral')   # placeholder nếu chưa có nhãn
    rows.append({'clip_id': clip_id, 'emotion_label': label, 'video_path': vp})

with open(LABELS_CSV, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['clip_id', 'emotion_label', 'video_path'])
    writer.writeheader()
    writer.writerows(rows)

print(f'Labels CSV: {LABELS_CSV}')
print(f'Số clip: {len(rows)}')
if any(r["emotion_label"] == "neutral" and r["clip_id"] not in CLIP_LABELS for r in rows):
    print('⚠️  Một số clip dùng nhãn placeholder "neutral" — hãy điền nhãn thực vào CLIP_LABELS')

## Bước 2 — Tiền xử lý video

In [ ]:
# ============================================================
# CELL 8 — Tách audio + frames
# ============================================================
# Kiểm tra ffmpeg
r = subprocess.run(['ffmpeg', '-version'], capture_output=True)
print('ffmpeg:', 'OK' if r.returncode == 0 else '❌ KHÔNG TÌM THẤY')

sys.path.insert(0, SRC_DIR) if SRC_DIR not in sys.path else None

from vie_gameemo.preprocess.demux import extract_audio, extract_frames
from pathlib import Path

for vp in video_paths:
    clip_id = os.path.splitext(os.path.basename(vp))[0]
    audio_dir = os.path.join(PROC_DIR, 'audios')
    frames_dir = os.path.join(PROC_DIR, 'frames', clip_id)
    os.makedirs(audio_dir, exist_ok=True)
    os.makedirs(frames_dir, exist_ok=True)

    audio_path = os.path.join(audio_dir, f'{clip_id}.wav')
    if not os.path.exists(audio_path):
        extract_audio(Path(vp), Path(audio_path))
        print(f'  Audio → {audio_path}')
    else:
        print(f'  Skip audio (đã có): {clip_id}')

    if not os.listdir(frames_dir):
        n = extract_frames(Path(vp), Path(frames_dir), target_fps=2)
        print(f'  Frames → {frames_dir} ({n} frames)')
    else:
        print(f'  Skip frames (đã có): {clip_id}')

print('\n✅ Tiền xử lý xong')

In [ ]:
# ============================================================
# CELL 9 — Phát hiện vùng webcam (MediaPipe DBSCAN)
# ============================================================
from vie_gameemo.preprocess.webcam_detector import WebcamDetector
import json

detector = WebcamDetector()
webcam_bboxes = {}

for vp in video_paths:
    clip_id = os.path.splitext(os.path.basename(vp))[0]
    bbox = detector.detect_webcam_region(Path(vp))
    if bbox:
        webcam_bboxes[clip_id] = bbox.model_dump()
        print(f'  {clip_id}: webcam tại x={bbox.x:.2f} y={bbox.y:.2f} w={bbox.w:.2f} h={bbox.h:.2f}')
    else:
        webcam_bboxes[clip_id] = None
        print(f'  {clip_id}: ⚠️  Không phát hiện webcam')

bbox_path = os.path.join(PROC_DIR, 'webcam_bboxes.json')
with open(bbox_path, 'w') as f:
    json.dump(webcam_bboxes, f, indent=2)
print(f'\nWebcam bboxes → {bbox_path}')

## Bước 3 — Annotation Pipeline

Mỗi agent chạy tuần tự: **tải → xử lý toàn bộ → unload** để tiết kiệm VRAM (T4 16GB).

| Agent | Model | VRAM | Thời gian |  
|-------|-------|------|----------|
| Whisper ASR | whisper-large-v3 | ~3 GB | ~30s/clip |
| OpenFace AUs | binary | CPU | ~10s/clip |
| Qwen-VL | VL-7B 4bit | ~7 GB | ~20s/clip |
| Qwen-Audio | Audio-7B 4bit | ~5 GB | ~15s/clip |
| Consolidator | 7B 4bit | ~5 GB | ~30s/clip |

In [ ]:
# ============================================================
# CELL 10 — Phase 1: Whisper ASR (transcript)
# ============================================================
from vie_gameemo.data.annotator.whisper_asr import WhisperASR

print('Loading Whisper large-v3...')
asr = WhisperASR(
    model_name='openai/whisper-large-v3',
    compute_type='float16',
    language='vi',
    vad_filter=True,
)
asr.load()

transcripts = {}
audio_dir = os.path.join(PROC_DIR, 'audios')
for vp in video_paths:
    clip_id = os.path.splitext(os.path.basename(vp))[0]
    audio_path = os.path.join(audio_dir, f'{clip_id}.wav')
    if os.path.exists(audio_path):
        text = asr.transcribe(Path(audio_path))
        transcripts[clip_id] = text
        print(f'  {clip_id}: "{text[:80]}..."' if len(text) > 80 else f'  {clip_id}: "{text}"')
    else:
        transcripts[clip_id] = ''
        print(f'  {clip_id}: ⚠️  audio không tìm thấy')

# Unload Whisper
del asr
import gc; gc.collect()
torch.cuda.empty_cache()
print('\n✅ Whisper unloaded')

In [ ]:
# ============================================================
# CELL 11 — Phase 2: AU Extraction (OpenFace hoặc MediaPipe fallback)
# ============================================================
au_results = {}

if USE_OPENFACE:
    from vie_gameemo.data.annotator.openface_au import extract_aus, aggregate_au_intensity
    au_tmp = os.path.join(WORKING, 'openface_tmp')
    os.makedirs(au_tmp, exist_ok=True)
    for vp in video_paths:
        clip_id = os.path.splitext(os.path.basename(vp))[0]
        try:
            aus = extract_aus(Path(vp), OPENFACE_BINARY, Path(au_tmp))
            intensity = aggregate_au_intensity(aus)
            # Tóm tắt thành string cho LLM
            top_aus = sorted(aus.items(), key=lambda x: -max(x[1]) if x[1] else 0)[:5]
            au_str = ', '.join(f'AU{k}={max(v):.1f}' for k, v in top_aus if v)
            au_results[clip_id] = au_str or 'N/A'
            print(f'  {clip_id}: {au_str}')
        except Exception as e:
            au_results[clip_id] = 'N/A'
            print(f'  {clip_id}: ⚠️  {e}')
else:
    # Fallback: ước lượng AU từ MediaPipe Face Mesh
    print('Dùng MediaPipe fallback cho AU estimation...')
    import mediapipe as mp
    import cv2
    import numpy as np

    mp_face = mp.solutions.face_mesh

    for vp in video_paths:
        clip_id = os.path.splitext(os.path.basename(vp))[0]
        frames_dir = os.path.join(PROC_DIR, 'frames', clip_id)
        frame_files = sorted(f for f in os.listdir(frames_dir) if f.endswith('.jpg'))[:8]

        au_desc = 'N/A'
        if frame_files:
            # Đọc frame giữa clip
            mid_frame = cv2.imread(os.path.join(frames_dir, frame_files[len(frame_files)//2]))
            mid_frame_rgb = cv2.cvtColor(mid_frame, cv2.COLOR_BGR2RGB)
            with mp_face.FaceMesh(static_image_mode=True, max_num_faces=1) as face_mesh:
                results = face_mesh.process(mid_frame_rgb)
                if results.multi_face_landmarks:
                    au_desc = 'Face detected (landmarks OK, no AU values — OpenFace needed for exact AUs)'
                else:
                    au_desc = 'No face detected'
        au_results[clip_id] = au_desc
        print(f'  {clip_id}: {au_desc}')

print('\n✅ AU extraction xong')

In [ ]:
# ============================================================
# CELL 12 — Phase 3: Qwen-VL Visual Descriptions
# ============================================================
from vie_gameemo.data.annotator.qwen_vl_agent import QwenVLAgent

print(f'Loading Qwen-VL: {QWEN_VL_MODEL}...')
vl_agent = QwenVLAgent(
    model_name=QWEN_VL_MODEL,
    quantization=QUANTIZATION,
)
vl_agent.load()

visual_descriptions = {}
for vp in video_paths:
    clip_id = os.path.splitext(os.path.basename(vp))[0]
    frames_dir = os.path.join(PROC_DIR, 'frames', clip_id)
    frame_files = sorted(
        [Path(frames_dir) / f for f in os.listdir(frames_dir) if f.endswith('.jpg')]
    )[:4]   # Dùng 4 frames để mô tả

    if frame_files:
        try:
            descs = vl_agent.batch_describe(frame_files)
            visual_descriptions[clip_id] = ' | '.join(descs)
            print(f'  {clip_id}: "{descs[0][:100]}..."')
        except Exception as e:
            visual_descriptions[clip_id] = 'N/A'
            print(f'  {clip_id}: ⚠️  {e}')
    else:
        visual_descriptions[clip_id] = 'N/A'

# Unload
vl_agent.unload()
del vl_agent
gc.collect(); torch.cuda.empty_cache()
print('\n✅ Qwen-VL unloaded')

In [ ]:
# ============================================================
# CELL 13 — Phase 4: Qwen-Audio Descriptions
# ============================================================
from vie_gameemo.data.annotator.qwen_audio_agent import QwenAudioAgent

print(f'Loading Qwen-Audio: {QWEN_AUDIO_MODEL}...')
audio_agent = QwenAudioAgent(
    model_name=QWEN_AUDIO_MODEL,
    quantization=QUANTIZATION,
)
audio_agent.load()

audio_descriptions = {}
audio_dir = os.path.join(PROC_DIR, 'audios')
for vp in video_paths:
    clip_id = os.path.splitext(os.path.basename(vp))[0]
    audio_path = Path(os.path.join(audio_dir, f'{clip_id}.wav'))
    if audio_path.exists():
        try:
            desc = audio_agent.batch_describe([audio_path])[0]
            audio_descriptions[clip_id] = desc
            print(f'  {clip_id}: "{desc[:100]}..."')
        except Exception as e:
            audio_descriptions[clip_id] = 'N/A'
            print(f'  {clip_id}: ⚠️  {e}')
    else:
        audio_descriptions[clip_id] = 'N/A'

# Unload
audio_agent.unload()
del audio_agent
gc.collect(); torch.cuda.empty_cache()
print('\n✅ Qwen-Audio unloaded')

In [ ]:
# ============================================================
# CELL 14 — Phase 5: Consolidator (tổng hợp reasoning)
# ============================================================
from vie_gameemo.data.annotator.consolidator import Consolidator

# Hỗ trợ cả Qwen2.5 và Qwen3
ENABLE_THINKING = 'Qwen3' in ANNOTATION_MODEL   # Qwen3 hỗ trợ /think token
print(f'Loading Consolidator: {ANNOTATION_MODEL} (thinking={ENABLE_THINKING})...')

consolidator = Consolidator(
    model_name=ANNOTATION_MODEL,
    quantization=QUANTIZATION,
)
consolidator.load()

reasoning_results = {}
import csv
label_map = {r['clip_id']: r['emotion_label'] for r in csv.DictReader(open(LABELS_CSV))}

for vp in video_paths:
    clip_id = os.path.splitext(os.path.basename(vp))[0]
    evidence = {
        'emotion_label': label_map.get(clip_id, 'neutral'),
        'face_aus': au_results.get(clip_id, 'N/A'),
        'visual_objective': visual_descriptions.get(clip_id, 'N/A'),
        'audio_tone': audio_descriptions.get(clip_id, 'N/A'),
        'transcript': transcripts.get(clip_id, ''),
    }
    try:
        result = consolidator.consolidate(evidence)
        reasoning_results[clip_id] = result
        print(f'  {clip_id} [{evidence["emotion_label"]}]: {result.get("reasoning","")[:80]}...')
    except Exception as e:
        reasoning_results[clip_id] = {'reasoning': '', 'answer': evidence['emotion_label']}
        print(f'  {clip_id}: ⚠️  {e}')

consolidator.unload()
del consolidator
gc.collect(); torch.cuda.empty_cache()
print('\n✅ Consolidation xong')

## Bước 4 — Lưu Annotations

In [ ]:
# ============================================================
# CELL 15 — Tạo Annotation JSON cho từng clip
# ============================================================
import json
from datetime import datetime

saved = []
for vp in video_paths:
    clip_id = os.path.splitext(os.path.basename(vp))[0]
    reasoning_data = reasoning_results.get(clip_id, {})

    annotation = {
        'clip_id': clip_id,
        'video_path': vp,
        'emotion_label': label_map.get(clip_id, 'neutral'),
        'transcript': transcripts.get(clip_id, ''),
        'face_aus': au_results.get(clip_id, 'N/A'),
        'visual_objective': visual_descriptions.get(clip_id, 'N/A'),
        'audio_tone': audio_descriptions.get(clip_id, 'N/A'),
        'reasoning': reasoning_data.get('reasoning', ''),
        'reasoning_label': reasoning_data.get('answer', label_map.get(clip_id, 'neutral')),
        'annotation_model': ANNOTATION_MODEL,
        'created_at': datetime.now().isoformat(),
        'split': 'train',   # Sẽ được phân chia lại trong training notebook
    }

    out_path = os.path.join(ANNOT_DIR, f'{clip_id}.json')
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(annotation, f, ensure_ascii=False, indent=2)
    saved.append(out_path)

print(f'✅ Đã lưu {len(saved)} annotation files → {ANNOT_DIR}')
for p in saved[:3]:
    print(f'  {p}')

In [ ]:
# ============================================================
# CELL 16 — Kiểm tra + thống kê
# ============================================================
from collections import Counter

annotations = []
for f in os.listdir(ANNOT_DIR):
    if f.endswith('.json'):
        with open(os.path.join(ANNOT_DIR, f), encoding='utf-8') as fp:
            annotations.append(json.load(fp))

print(f'Tổng annotations: {len(annotations)}')
label_counts = Counter(a['emotion_label'] for a in annotations)
print('\nPhân phối nhãn:')
for label in VALID_LABELS:
    count = label_counts.get(label, 0)
    bar = '█' * count
    print(f'  {label:<15}: {bar} ({count})')

has_reasoning = sum(1 for a in annotations if a.get('reasoning'))
has_transcript = sum(1 for a in annotations if a.get('transcript'))
print(f'\nCó reasoning : {has_reasoning}/{len(annotations)}')
print(f'Có transcript: {has_transcript}/{len(annotations)}')

In [ ]:
# ============================================================
# CELL 17 — Tạo archive để download
# ============================================================
import zipfile

archive_path = os.path.join(WORKING, 'stage0_annotations.zip')
with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in os.listdir(ANNOT_DIR):
        zf.write(os.path.join(ANNOT_DIR, f), f'annotations/{f}')
    zf.write(LABELS_CSV, 'annotations/labels.csv')
    # Thêm webcam bboxes
    bbox_file = os.path.join(PROC_DIR, 'webcam_bboxes.json')
    if os.path.exists(bbox_file):
        zf.write(bbox_file, 'processed/webcam_bboxes.json')

print(f'✅ Archive: {archive_path}')
print(f'   Size: {os.path.getsize(archive_path) / 1e6:.1f} MB')
print()
print('📥 Để download: File browser (bên trái) → tìm stage0_annotations.zip → chuột phải → Download')
print('   Hoặc thêm /kaggle/working vào Kaggle output dataset để dùng trong notebook tiếp theo')